# Explore FoodUSDA — Latest vs Legacy

**Mission**: Data exploratory on the FoodUSDA dataset stored in `/workspace/ICOCO.duckdb`, using **only** the silver layer. Two versions exist — `silver.FoodUSDA_latest_*` and `silver.FoodUSDA_legacy_*` — so a **comparison** is a core part of this exploration.

**Source**: built by [`../1_silver_create/silver_curation.ipynb`](../1_silver_create/silver_curation.ipynb). The silver tables keep only the join-critical columns:

| Silver table | Columns |
|---|---|
| `*_food` | `fdc_id, data_type, description, food_category_id` |
| `*_food_category` | `id, code, description` |
| `*_nutrient` | `id, name` |
| `*_food_nutrient` | `id, fdc_id, nutrient_id` |

> ⚠️ **Important limitation**: the silver `*_food_nutrient` table keeps only the **incidence** (`fdc_id ↔ nutrient_id`). The original USDA `food_nutrient` also carries `amount / unit / min / max …` — those **values were dropped** in the silver curation. So we can analyse *which nutrient a food lists*, but **not how much** of it.

**Principles**
- **SQL-first**, **read-only** (no tables/views/schemas created), **silver only**.

## 0. Setup (connect to DuckDB)

Same connection pattern as the silver notebook — one DuckDB connection at a time.

In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 100
%sql duckdb:////workspace/ICOCO.duckdb

Connecting to 'duckdb:////workspace/ICOCO.duckdb'

In [2]:
# Close then reconnect to flush the WAL file
%sql --close duckdb:////workspace/ICOCO.duckdb

In [3]:
%sql duckdb:////workspace/ICOCO.duckdb

Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

## 1. Overview — scale of each version

How many foods, nutrient definitions and food↔nutrient incidence rows does each version hold?

In [4]:
%%sql
SELECT 'latest' AS version,
       (SELECT COUNT(*) FROM silver.FoodUSDA_latest_food)                    AS n_food,
       (SELECT COUNT(*) FROM silver.FoodUSDA_latest_nutrient)                AS n_nutrient_defs,
       (SELECT COUNT(*) FROM silver.FoodUSDA_latest_food_nutrient)           AS n_food_nutrient_rows,
       (SELECT COUNT(DISTINCT fdc_id) FROM silver.FoodUSDA_latest_food_nutrient) AS n_food_with_nutrients
UNION ALL
SELECT 'legacy',
       (SELECT COUNT(*) FROM silver.FoodUSDA_legacy_food),
       (SELECT COUNT(*) FROM silver.FoodUSDA_legacy_nutrient),
       (SELECT COUNT(*) FROM silver.FoodUSDA_legacy_food_nutrient),
       (SELECT COUNT(DISTINCT fdc_id) FROM silver.FoodUSDA_legacy_food_nutrient);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

version,n_food,n_nutrient_defs,n_food_nutrient_rows,n_food_with_nutrients
latest,87990,477,170469,76036
legacy,7793,474,644125,7793


## 2. Latest — deep dive

### 2.1 `data_type` distribution

`data_type` describes *how* a food record was acquired (branded products, lab samples, foundation foods, …).

In [5]:
%%sql
SELECT data_type,
       COUNT(*)                       AS n_food
FROM silver.FoodUSDA_latest_food
GROUP BY data_type
ORDER BY n_food DESC;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

data_type,n_food
sub_sample_food,75055
market_acquisition,7577
sample_food,4079
agricultural_acquisition,810
foundation_food,469


**Read**: the `latest` release is dominated by branded `sub_sample_food` (75k, ~85%) — commercial packaged foods. Only 469 `foundation_food` records exist.

### 2.2 Nutrient coverage per `data_type`

How much nutrition data does each *acquisition type* actually carry? `avg_per_food = nutrient rows / distinct foods` for that type.

In [6]:
%%sql
SELECT f.data_type,
       COUNT(DISTINCT f.fdc_id)                          AS n_food,
       COUNT(fn.fdc_id)                                  AS n_fn_rows,
       ROUND(COUNT(fn.fdc_id) * 1.0 /
             COUNT(DISTINCT f.fdc_id), 2)                AS avg_nutrient_rows_per_food
FROM   silver.FoodUSDA_latest_food f
LEFT JOIN silver.FoodUSDA_latest_food_nutrient fn USING (fdc_id)
GROUP BY f.data_type
ORDER BY avg_nutrient_rows_per_food DESC;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

data_type,n_food,n_fn_rows,avg_nutrient_rows_per_food
foundation_food,469,21459,45.75
agricultural_acquisition,810,14574,17.99
sub_sample_food,75055,134436,1.79
market_acquisition,7577,0,0.0
sample_food,4079,0,0.0


**Read**: `foundation_food` is nutritionally dense (~46 rows/food) while `market_acquisition` and `sample_food` contribute **zero** food_nutrient rows in the silver extract. The thousands of branded `sub_sample_food` carry a thin ~1.8 rows/food.

### 2.3 Top food categories

In [7]:
%%sql
SELECT COALESCE(fc.description, '(no category)') AS category,
       COUNT(*)                              AS n_food
FROM silver.FoodUSDA_latest_food f
LEFT JOIN silver.FoodUSDA_latest_food_category fc ON f.food_category_id = fc.id
GROUP BY fc.description
ORDER BY n_food DESC
LIMIT 12;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

category,n_food
Vegetables and Vegetable Products,17641
Fruits and Fruit Juices,12448
Dairy and Egg Products,10970
Legumes and Legume Products,8750
Cereal Grains and Pasta,6578
Fats and Oils,4391
Sausages and Luncheon Meats,4115
Finfish and Shellfish Products,4114
Nut and Seed Products,4010
(no category),3811


**Read**: Vegetables/Fruits/Dairy lead; foods with no category number 3,811 (≈4%).

### 2.4 Most frequently recorded nutrients

In [8]:
%%sql
SELECT n.name,
       COUNT(*) AS n_records
FROM silver.FoodUSDA_latest_food_nutrient fn
JOIN silver.FoodUSDA_latest_nutrient n ON fn.nutrient_id = n.id
GROUP BY n.name
ORDER BY n_records DESC
LIMIT 12;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

name,n_records
Water,5122
Total lipid (fat),4756
"Copper, Cu",4725
"Magnesium, Mg",4725
"Manganese, Mn",4724
"Phosphorus, P",4724
"Zinc, Zn",4723
"Calcium, Ca",4723
"Iron, Fe",4719
"Potassium, K",4712


## 3. Legacy — deep dive

### 3.1 `data_type` distribution

In [9]:
%%sql
SELECT data_type,
       COUNT(*) AS n_food
FROM silver.FoodUSDA_legacy_food
GROUP BY data_type;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

data_type,n_food
sr_legacy_food,7793


**Read**: the legacy release is homogeneous — every row is `sr_legacy_food` (the classic USDA Standard Reference legacy set).

### 3.2 Nutrient coverage

For the legacy set the nutrition data is dense and nearly universal:

In [10]:
%%sql
SELECT COUNT(DISTINCT f.fdc_id)                               AS n_food,
       COUNT(DISTINCT fn.fdc_id)                              AS n_food_with_nutrients,
       ROUND(COUNT(fn.fdc_id) * 1.0 /
             COUNT(DISTINCT f.fdc_id), 2)                     AS avg_nutrient_rows_per_food
FROM       silver.FoodUSDA_legacy_food f
LEFT JOIN  silver.FoodUSDA_legacy_food_nutrient fn USING (fdc_id);

Running query in 'duckdb:////workspace/ICOCO.duckdb'

n_food,n_food_with_nutrients,avg_nutrient_rows_per_food
7793,7793,82.65


**Read**: every legacy food (~100%) has nutrient rows, ~82 per food on average — a complete curated reference profile.

### 3.3 Top food categories

In [11]:
%%sql
SELECT COALESCE(fc.description, '(no category)') AS category,
       COUNT(*)                              AS n_food
FROM silver.FoodUSDA_legacy_food f
LEFT JOIN silver.FoodUSDA_legacy_food_category fc ON f.food_category_id = fc.id
GROUP BY fc.description
ORDER BY n_food DESC
LIMIT 12;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

category,n_food
Beef Products,954
Vegetables and Vegetable Products,814
Baked Products,517
"Lamb, Veal, and Game Products",464
Poultry Products,383
Beverages,366
Sweets,358
Fruits and Fruit Juices,355
Baby Foods,345
Pork Products,336


**Read**: legacy has **no** NULL categories (all mapped) and a flatter, reference-oriented distribution.

### 3.4 Most frequently recorded nutrients

In [12]:
%%sql
SELECT n.name,
       COUNT(*) AS n_records
FROM silver.FoodUSDA_legacy_food_nutrient fn
JOIN silver.FoodUSDA_legacy_nutrient n ON fn.nutrient_id = n.id
GROUP BY n.name
ORDER BY n_records DESC
LIMIT 12;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

name,n_records
Energy,15586
Total lipid (fat),7793
"Carbohydrate, by difference",7793
Protein,7793
Water,7793
Ash,7790
"Iron, Fe",7713
"Sodium, Na",7709
"Calcium, Ca",7708
"Potassium, K",7516


**Read**: `Energy` / `Protein` / `Water` / `Total lipid (fat)` appear on essentially every legacy food — the classic macros. (`Energy` has ~15.6k records across 7.8k foods because multiple *energy* nutrient definitions exist, e.g. different Atwater adjustment factors.)

## 4. Latest vs Legacy — side by side

In [13]:
%%sql
SELECT 'latest' AS version,
       (SELECT COUNT(*) FROM silver.FoodUSDA_latest_food)                            AS n_food,
       (SELECT COUNT(*) FROM silver.FoodUSDA_latest_food WHERE food_category_id IS NULL) AS n_food_no_cat,
       (SELECT COUNT(*) FROM silver.FoodUSDA_latest_nutrient)                        AS n_nutrient_defs,
       (SELECT COUNT(DISTINCT nutrient_id) FROM silver.FoodUSDA_latest_food_nutrient) AS n_nutrient_used,
       (SELECT COUNT(*) FROM silver.FoodUSDA_latest_food_nutrient)                   AS n_fn_rows,
       (SELECT ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT fdc_id), 2) FROM silver.FoodUSDA_latest_food_nutrient) AS avg_fn_rows_per_food
UNION ALL
SELECT 'legacy',
       (SELECT COUNT(*) FROM silver.FoodUSDA_legacy_food),
       (SELECT COUNT(*) FROM silver.FoodUSDA_legacy_food WHERE food_category_id IS NULL),
       (SELECT COUNT(*) FROM silver.FoodUSDA_legacy_nutrient),
       (SELECT COUNT(DISTINCT nutrient_id) FROM silver.FoodUSDA_legacy_food_nutrient),
       (SELECT COUNT(*) FROM silver.FoodUSDA_legacy_food_nutrient),
       (SELECT ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT fdc_id), 2) FROM silver.FoodUSDA_legacy_food_nutrient)
ORDER BY version;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

version,n_food,n_food_no_cat,n_nutrient_defs,n_nutrient_used,n_fn_rows,avg_fn_rows_per_food
latest,87990,3811,477,235,170469,2.24
legacy,7793,0,474,149,644125,82.65


How much do the two versions' nutrient vocabularies overlap at the incidence level?

In [14]:
%%sql
SELECT COUNT(*) AS n_shared_nutrient_ids
FROM (
    SELECT DISTINCT nutrient_id FROM silver.FoodUSDA_latest_food_nutrient
    INTERSECT
    SELECT DISTINCT nutrient_id FROM silver.FoodUSDA_legacy_food_nutrient
) t;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

n_shared_nutrient_ids
134


## 5. Summary / findings worth investigating

| Dimension | Latest | Legacy |
|-----------|--------|--------|
| Foods | **87,990** | 7,793 |
| `data_type` | 5 types (dominated by branded `sub_sample_food`) | 1 (`sr_legacy_food`) |
| Foods without category | 3,811 (≈4%) | 0 |
| Nutrient definitions | 477 | 474 |
| Nutrients actually used | 235 | 149 |
| food↔nutrient rows | 170,469 | 644,125 |
| **Avg nutrient rows per food** | **~2.2** | **~82.7** |

**Worth investigating**
- **The two versions are complementary, not interchangeable.** `legacy` = small, dense, curated reference (SR Legacy) with near-complete profiles. `latest` = large, broad catalog (mostly branded foods) with **sparse** nutrient data.
- **Ring the alarm on the `<amount>` drop in silver.** Because `silver.*_food_nutrient` kept only `fdc_id, nutrient_id` (no `amount`/`unit`), you cannot compute actual nutrient intakes from the silver layer — only which nutrient is present. If an analysis needs quantities, the silver schema must be extended.
- **In the latest release, most "food coverage" is illusory for nutrition:** ≥85% of foods are branded with ~1.8 rows/food, and `market_acquisition`/`sample_food` have **0** nutrient rows here. Only `foundation_food`/`agricultural_acquisition` are nutritionally comprehensive.
- **Category coverage differs:** legacy maps every food to a category; latest leaves ~4% unmapped.